# Bank Loan Default Prediction
### Final Project – Supervised Machine Learning (Classification)

**Dataset Source:** Kaggle – [Bank Loan Status Dataset](https://www.kaggle.com/datasets/zaurbegiev/my-dataset)

---

## Problem Definition

When a bank gives a loan to someone, there is always a risk that the person might not pay it back. This is called **loan default**. In this project, I will try to build a machine learning model that can predict whether a person will **default on their loan or not** based on their information like income, credit score, loan amount etc.

This is a **Binary Classification** problem:
- **1** = Loan Defaulted (person did not pay back)
- **0** = Loan Fully Paid

---

## Step 1: Import Libraries

First I will import all the libraries I need for this project.

In [ ]:
# basic libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# for machine learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve

print("All libraries imported successfully!")

## Step 2: Load the Dataset


In [1]:
df = pd.read_csv('credit_train.csv')

print("Dataset loaded!")
print("Shape of dataset:", df.shape)

NameError: name 'pd' is not defined

## Step 3: Data Understanding



In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# check the target variable - Loan Status
print("Target variable distribution:")
print(df['Loan Status'].value_counts())
print()
print("Percentage:")
print(df['Loan Status'].value_counts(normalize=True) * 100)

## Step 4: Data Preprocessing

Now I will clean the data. This includes:
- Checking for missing values
- Handling missing values
- Encoding categorical variables
- Feature Scaling

In [ ]:
print("Missing values in each column:")
print(df.isnull().sum())
print()
print("Total missing values:", df.isnull().sum().sum())

In [ ]:
plt.figure(figsize=(12, 5))
sns.heatmap(df.isnull(), yticklabels=False, cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
threshold = 0.4
missing_percent = df.isnull().mean()
cols_to_drop = missing_percent[missing_percent > threshold].index.tolist()
print("Columns dropped due to high missing values:", cols_to_drop)
df.drop(columns=cols_to_drop, inplace=True)

In [ ]:
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

#fill missing categorical values with mode
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

print("Missing values after filling:")
print(df.isnull().sum().sum())

In [ ]:
#encode the target variable
df['Loan Status'] = df['Loan Status'].map({'Charged Off': 1, 'Fully Paid': 0})

print("Target variable after encoding:")
print(df['Loan Status'].value_counts())

In [ ]:
#encode categorical columns using Label Encoder
le = LabelEncoder()

cat_cols = df.select_dtypes(include=['object']).columns
print("Categorical columns to encode:", list(cat_cols))
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))
print("\nEncoding done!")
df.head()

In [ ]:
#check for outliers using boxplot
important_cols = ['Annual Income', 'Current Loan Amount', 'Credit Score', 'Monthly Debt']

#only use columns that exist in dataset
important_cols = [c for c in important_cols if c in df.columns]

plt.figure(figsize=(14, 6))
for i, col in enumerate(important_cols):
    plt.subplot(1, len(important_cols), i+1)
    sns.boxplot(y=df[col])
    plt.title(col)
plt.tight_layout()
plt.suptitle('Outlier Detection using Boxplots', y=1.02, fontsize=14)
plt.show()

In [ ]:
#remove outliers using IQR method
def remove_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 3 * IQR
    upper = Q3 + 3 * IQR
    return df[(df[col] >= lower) & (df[col] <= upper)]

original_size = len(df)
for col in important_cols:
    df = remove_outliers(df, col)
print(f"Rows before outlier removal: {original_size}")
print(f"Rows after outlier removal: {len(df)}")

## Step 5: Exploratory Data Analysis (EDA)


### Univariate Analysis

In [ ]:
#distribution of target variable
plt.figure(figsize=(6, 4))
df['Loan Status'].value_counts().plot(kind='bar', color=['steelblue', 'tomato'])
plt.title('Loan Status Distribution')
plt.xlabel('Loan Status (0=Fully Paid, 1=Defaulted)')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

In [ ]:
#distribution of numerical features
num_features = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
num_features = [f for f in num_features if f != 'Loan Status']

#plot first 6 numerical features
plot_cols = num_features[:6]

plt.figure(figsize=(15, 8))
for i, col in enumerate(plot_cols):
    plt.subplot(2, 3, i+1)
    sns.histplot(df[col], kde=True, color='steelblue')
    plt.title(f'Distribution of {col}')
plt.tight_layout()
plt.show()

### 5.2 Bivariate Analysis

In [ ]:
#compare important features with loan status
plt.figure(figsize=(15, 8))
for i, col in enumerate(plot_cols):
    plt.subplot(2, 3, i+1)
    sns.boxplot(x='Loan Status', y=col, data=df, palette=['steelblue', 'tomato'])
    plt.title(f'{col} vs Loan Status')
plt.tight_layout()
plt.suptitle('Feature vs Loan Status (Bivariate Analysis)', y=1.02, fontsize=14)
plt.show()

In [ ]:
#if credit score column exists, let's look at it more carefully
if 'Credit Score' in df.columns:
    plt.figure(figsize=(8, 5))
    sns.kdeplot(data=df, x='Credit Score', hue='Loan Status', fill=True)
    plt.title('Credit Score Distribution by Loan Status')
    plt.xlabel('Credit Score')
    plt.show()
    print("People who defaulted tend to have lower credit scores.")

### 5.3 Multivariate Analysis

In [ ]:
# correlation matrix
plt.figure(figsize=(12, 8))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()
print("Correlation with Loan Status:")
print(corr['Loan Status'].sort_values(ascending=False))

In [ ]:
#pairplot of top features
top_features = corr['Loan Status'].abs().sort_values(ascending=False)[1:5].index.tolist()
top_features.append('Loan Status')

pairplot_df = df[top_features].sample(min(500, len(df)))
sns.pairplot(pairplot_df, hue='Loan Status', palette=['steelblue', 'tomato'])
plt.suptitle('Pairplot of Top Features', y=1.02)
plt.show()

## Step 6: Feature Engineering & Feature Selection


In [ ]:
# separate features and target
X = df.drop('Loan Status', axis=1)
y = df['Loan Status']

print("Features shape:", X.shape)
print("Target shape:", y.shape)

In [2]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)

NameError: name 'train_test_split' is not defined

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling done!")

## Step 7: Model Building

I will train 3 different models and compare them:
1. Logistic Regression
2. Decision Tree
3. Random Forest

### Model 1: Logistic Regression

In [ ]:
# train logistic regression model
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

# predict
lr_pred = lr_model.predict(X_test_scaled)
lr_prob = lr_model.predict_proba(X_test_scaled)[:, 1]

# evaluate
lr_acc = accuracy_score(y_test, lr_pred)
lr_auc = roc_auc_score(y_test, lr_prob)

print("=== Logistic Regression Results ===")
print(f"Accuracy: {lr_acc:.4f}")
print(f"ROC-AUC: {lr_auc:.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, lr_pred))

In [ ]:
# confusion matrix for logistic regression
plt.figure(figsize=(6, 4))
cm = confusion_matrix(y_test, lr_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Logistic Regression')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

### Model 2: Decision Tree

In [ ]:
# train decision tree model
# I will limit the depth to avoid overfitting
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)

# predict
dt_pred = dt_model.predict(X_test)
dt_prob = dt_model.predict_proba(X_test)[:, 1]

# evaluate
dt_acc = accuracy_score(y_test, dt_pred)
dt_auc = roc_auc_score(y_test, dt_prob)

print("=== Decision Tree Results ===")
print(f"Accuracy: {dt_acc:.4f}")
print(f"ROC-AUC: {dt_auc:.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, dt_pred))

In [ ]:
# confusion matrix for decision tree
plt.figure(figsize=(6, 4))
cm2 = confusion_matrix(y_test, dt_pred)
sns.heatmap(cm2, annot=True, fmt='d', cmap='Greens')
plt.title('Confusion Matrix - Decision Tree')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

### Model 3: Random Forest

In [ ]:
# train random forest model
# random forest is ensemble of many decision trees
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# predict
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

# evaluate
rf_acc = accuracy_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_prob)

print("=== Random Forest Results ===")
print(f"Accuracy: {rf_acc:.4f}")
print(f"ROC-AUC: {rf_auc:.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, rf_pred))

In [ ]:
# confusion matrix for random forest
plt.figure(figsize=(6, 4))
cm3 = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm3, annot=True, fmt='d', cmap='Oranges')
plt.title('Confusion Matrix - Random Forest')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## Step 8: Model Evaluation – ROC Curve

I will plot the ROC curve for all 3 models to compare them visually.

In [ ]:
# plot ROC curves for all models
plt.figure(figsize=(8, 6))

# logistic regression
fpr1, tpr1, _ = roc_curve(y_test, lr_prob)
plt.plot(fpr1, tpr1, label=f'Logistic Regression (AUC = {lr_auc:.2f})', color='blue')

# decision tree
fpr2, tpr2, _ = roc_curve(y_test, dt_prob)
plt.plot(fpr2, tpr2, label=f'Decision Tree (AUC = {dt_auc:.2f})', color='green')

# random forest
fpr3, tpr3, _ = roc_curve(y_test, rf_prob)
plt.plot(fpr3, tpr3, label=f'Random Forest (AUC = {rf_auc:.2f})', color='orange')

# diagonal line (random classifier)
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.grid(True)
plt.show()

## Step 9: Model Comparison

Let me compare all 3 models together in a table.

In [ ]:
# create comparison table
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest'],
    'Accuracy': [lr_acc, dt_acc, rf_acc],
    'ROC-AUC': [lr_auc, dt_auc, rf_auc]
})

results = results.sort_values('Accuracy', ascending=False)
results = results.reset_index(drop=True)
print(results.to_string(index=False))

In [ ]:
# bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# accuracy comparison
axes[0].bar(results['Model'], results['Accuracy'], color=['steelblue', 'green', 'orange'])
axes[0].set_title('Accuracy Comparison')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim([0, 1])
axes[0].tick_params(axis='x', rotation=15)
for i, v in enumerate(results['Accuracy']):
    axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center')

# AUC comparison
axes[1].bar(results['Model'], results['ROC-AUC'], color=['steelblue', 'green', 'orange'])
axes[1].set_title('ROC-AUC Comparison')
axes[1].set_ylabel('ROC-AUC Score')
axes[1].set_ylim([0, 1])
axes[1].tick_params(axis='x', rotation=15)
for i, v in enumerate(results['ROC-AUC']):
    axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center')

plt.tight_layout()
plt.show()

In [ ]:
# feature importance from random forest
# this shows which features are most important for prediction
feature_imp = pd.Series(rf_model.feature_importances_, index=X.columns)
feature_imp = feature_imp.sort_values(ascending=False)[:10]  # top 10 features

plt.figure(figsize=(10, 6))
feature_imp.plot(kind='barh', color='steelblue')
plt.title('Top 10 Important Features (Random Forest)')
plt.xlabel('Importance Score')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:

print("=" * 50)
print("FINAL PROJECT SUMMARY")
print("=" * 50)
print(f"Project: Bank Loan Default Prediction")
print(f"Problem Type: Binary Classification")
print(f"Total Samples Used: {len(df)}")
print(f"Number of Features: {X.shape[1]}")
print()
print("Model Performance:")
for _, row in results.iterrows():
    print(f"  {row['Model']}: Accuracy={row['Accuracy']:.4f}, AUC={row['ROC-AUC']:.4f}")
print()
best_model = results.iloc[0]['Model']
print(f"Best Model: {best_model}")
print("=" * 50)